# Mesin Pencari — Versi Dasar (Baseline)

**Kelompok 5 · Sistem Temu Kembali Informasi**

Notebook ini adalah **Versi A (baseline)** tanpa query expansion. Sistem dasar
memakai BM25 + stopword removal + stemming Sastrawi. Notebook juga menampilkan
eksperimen ablasi TF-IDF/BM25 × stopword × stemming untuk menunjukkan
konfigurasi terbaik yang dipakai sebagai baseline.

> **Menjalankan di Google Colab:** pilih **Runtime → Run all**. Sel setup
> pertama menyiapkan modul `stki_utils.py` dan data final secara otomatis,
> jadi tidak perlu mengunggah file atau mengatur path.

Versi B (baseline + query expansion sinonim) ada di notebook
`mesin_pencari_updated_colab.ipynb`.


In [ ]:
#@title Setup data & modul (jalankan sekali) { display-mode: "form" }
# Sel ini membuat notebook mandiri: menulis stki_utils.py dan data final
# ke folder kerja, lalu menambahkan folder modul ke sys.path.
from pathlib import Path
import base64
import io
import sys
import zipfile

!pip -q --disable-pip-version-check install Sastrawi

BASE = Path("/content/stki_project") if Path("/content").exists() else Path.cwd() / "stki_project_colab"
BASE.mkdir(parents=True, exist_ok=True)

ZIP_B64 = """
...
"""

with zipfile.ZipFile(io.BytesIO(base64.b64decode(ZIP_B64))) as z:
    z.extractall(BASE)

sys.path.insert(0, str(BASE / "notebooks"))

import pandas as pd

korpus = pd.read_csv(BASE / "Data/korpus_final.csv")
queries = pd.read_csv(BASE / "Data/queries_25.csv")
qrels = pd.read_csv(BASE / "Data/qrels_final.csv")
assert (len(korpus), len(queries), len(qrels)) == (50, 25, 173), "Data tidak sesuai dataset final"
print(f"Setup selesai: {len(korpus)} dokumen, {len(queries)} query, {len(qrels)} qrels")
print(f"Modul dan data ada di: {BASE}")


In [2]:
import sys
from pathlib import Path

BASE = Path("/content/stki_project") if Path("/content/stki_project").exists() else Path.cwd() / "stki_project_colab"
if not (BASE / "notebooks" / "stki_utils.py").exists():
    raise FileNotFoundError(
        "Modul belum disiapkan. Jalankan sel setup di atas (Runtime > Run all)."
    )
sys.path.insert(0, str(BASE / "notebooks"))

import pandas as pd
pd.set_option("display.max_colwidth", 120)
import re
from stki_utils import (load_corpus, corpus_texts, preprocess, TfidfModel, BM25Model,
                        load_queries, load_qrels, evaluate)


class MesinPencari:
    def __init__(self, model="bm25", use_stopword=True, use_stemming=True):
        self.korpus = load_corpus().set_index("doc_id")
        self.opsi = dict(use_stopword=use_stopword, use_stemming=use_stemming)
        docs = {d: preprocess(t, **self.opsi) for d, t in corpus_texts().items()}
        self.model = BM25Model(docs) if model == "bm25" else TfidfModel(docs)

    def cari(self, query, k=10):
        return [d for d, _ in self.model.search(preprocess(query, **self.opsi), k=k)]

    def tampilkan(self, query, k=3):
        q_terms = set(preprocess(query, **self.opsi))
        print(f"Hasil pencarian untuk: {query!r}\n")
        for rank, (d, s) in enumerate(self.model.search(list(preprocess(query, **self.opsi)), k=k), 1):
            row = self.korpus.loc[d]
            kalimat = re.split(r"(?<=\.)\s+", row.isi)
            # cuplikan: kalimat dengan term query terbanyak
            cuplikan = max(kalimat, key=lambda s_: len(q_terms & set(preprocess(s_, **self.opsi))))
            print(f"{rank}. [{d}] {row.judul}  (skor {s:.3f})\n   ...{cuplikan}\n")


## Demo: Q03 (baseline)

Query Q03 (*strategi transformasi digital UMKM Indonesia*) dengan konfigurasi
baseline BM25 + stopword + stemming.


In [3]:
mesin = MesinPencari()
mesin.tampilkan("strategi transformasi digital UMKM Indonesia", k=5)


Hasil pencarian untuk: 'strategi transformasi digital UMKM Indonesia'

1. [K008] Segera Atasi Kendala dalam Transformasi Digital di Sektor UMKM  (skor 6.577)
   ...Artikel MPR menyoroti 70,2 persen UMKM menghadapi kendala saat bertransformasi menggunakan teknologi digital.

2. [K009] Satu Dekade, Transformasi Digital UMKM Dorong Pertumbuhan Ekonomi Nasional  (skor 6.478)
   ...Komdigi mencatat satu dekade transformasi digital telah mendorong pertumbuhan ekonomi nasional melalui UMKM.

3. [K048] Bappenas Dorong Inovasi Ekonomi Keuangan untuk Transformasi Digital  (skor 6.000)
   ...Bappenas mendorong inovasi ekonomi keuangan untuk mendukung transformasi digital.

4. [K004] Strategi Nasional KA Hadir untuk Tingkatkan Kesejahteraan dan Keamanan Digital Masyarakat  (skor 4.772)
   ...Komdigi menyatakan Strategi Nasional Kecerdasan Artifisial hadir untuk meningkatkan kesejahteraan dan keamanan digital masyarakat.

5. [K003] Strategi Nasional Kecerdasan Artifisial Indonesia 2020–2045  (skor 

## Evaluasi ablasi 25 query (k = 5)

Membandingkan TF-IDF/BM25 dan kombinasi stopword/stemming. Baris terakhir
(BM25 + stopword + stemming) adalah konfigurasi yang dipakai sebagai Versi A.


In [4]:
queries, qrels = load_queries(), load_qrels()
baris = []
for model in ["tfidf", "bm25"]:
    for sw in [False, True]:
        for st in [False, True]:
            m = MesinPencari(model, use_stopword=sw, use_stemming=st)
            run = {q: m.cari(t) for q, t in queries.items()}
            rata = evaluate(run, qrels, k=5).loc["Rata-rata"]
            baris.append({"model": model, "stopword": sw, "stemming": st,
                          "P@5": rata["P@5"], "MAP": rata["AP"], "nDCG@5": rata["nDCG@5"]})
pd.DataFrame(baris)

,model,stopword,stemming,P@5,MAP,nDCG@5
0,tfidf,False,False,0.856,0.8378,0.8463
1,tfidf,False,True,0.880,0.8485,0.8689
2,tfidf,True,False,0.872,0.8521,0.8544
3,tfidf,True,True,0.904,0.8714,0.8842
4,bm25,False,False,0.864,0.8645,0.8772
5,bm25,False,True,0.872,0.8704,0.8907
6,bm25,True,False,0.936,0.9072,0.9242
7,bm25,True,True,0.944,0.9103,0.9370


**Hasil.** Konfigurasi terbaik adalah **BM25 + stopword + stemming**:
P@5 0,944; MAP 0,9103; nDCG@5 0,9370. Konfigurasi ini menjadi **Versi A
(baseline)**. Notebook **Versi B** (`mesin_pencari_updated_colab.ipynb`)
menambahkan query expansion sinonim dan membandingkan A vs B; hasilnya
P@5 0,968; MAP 0,9295; nDCG@5 0,9613.
